# Frozen four-way comparison on fresh families, Kaggle T4

Same Tesla T4. Do not switch GPUs. Same Qwen3-4B revision, FP16, greedy decoding.

Uses `data/plan4_fresh.jsonl` (30 families, 90 variants). Do **not** load `plan4_dev`.

Unchanged procedures, same written A on every branch:

- `text`: compact RESULT lines.
- `text_full`: A's operational sheet plus that written working.
- `kv`: full completed cache.
- `probe_256`: question-conditioned headwise, sink 4 plus 256 selected positions (260 kept, about 36.56 MiB).

Controls: `no_message` and `mismatch` (compact text from another family).

Do not run budget 32/64 or OBF here. Those stay development diagnostics.

Development reference: compact text 77/90, fuller text 81/90, full KV 82/90, probe-256 41/90.

Download `/kaggle/working/results_stage_plan4_fresh`. Do not write into `results_stage_plan4_budget` or earlier plan4 folders.

In [ ]:
import os, json, shutil, sys
from pathlib import Path

os.environ["LATENT_RELAY_INFER_DTYPE"] = "fp16"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()

candidates = [Path.cwd()]
if Path("/kaggle/input").is_dir():
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in {".git", ".pytest_cache"}]
        if "stage_plan4_fresh.py" in files and Path(root).name == "src":
            candidates.append(Path(root).parent)

REPO = None
for cand in candidates:
    if (cand / "src" / "stage_plan4_fresh.py").is_file() and (cand / "data" / "plan4_fresh.jsonl").is_file():
        REPO = cand
        break
if REPO is None:
    raise FileNotFoundError("Need src/stage_plan4_fresh.py and data/plan4_fresh.jsonl.")

if ON_KAGGLE:
    dest = WORK / "latent-relay-audit"
    if str(REPO).startswith("/kaggle/input"):
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    elif not (dest / "src" / "stage_plan4_fresh.py").is_file():
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    REPO = dest
    os.chdir(REPO)
else:
    os.chdir(REPO)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("REPO", REPO.resolve())
print("plan4_fresh", sum(1 for _ in (REPO / "data" / "plan4_fresh.jsonl").open(encoding="utf-8")))

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.3", "tokenizers==0.22.2", "huggingface_hub==0.36.2",
    "safetensors==0.7.0", "accelerate==1.13.0", "datasets==4.4.1",
])
import torch, transformers
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("cuda", torch.cuda.is_available(), gpu)
print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.57")
if gpu is not None and "T4" not in gpu:
    raise RuntimeError(f"Stay on the locked Tesla T4. Got {gpu}.")

In [ ]:
subprocess.check_call([sys.executable, "scripts/fetch_upstream.py"])
print("Dest /kaggle/working/results_stage_plan4_fresh")
print("Frozen four-way plus no_message/mismatch. No 32/64 or OBF. plan4_fresh only.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_fresh", "--split", "plan4_fresh"])
print("plan4 fresh exit", rc)

In [ ]:
src = WORK / "results_stage_plan4_fresh"
if not (src / "summary.json").is_file():
    src = REPO / "results" / "stage_plan4_fresh" / "run"
print("reading", src)
path = src / "summary.json"
if path.is_file():
    print(path.read_text(encoding="utf-8")[:16000])